# Feature Engineering

This notebook prepares the cleaned Olist-style order data for downstream modeling by creating customer, seller, product, time-based, and delivery-quality features.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists() and (candidate / "raw data").exists():
            return candidate
    return start


project_root = find_project_root(Path.cwd().resolve())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

DATA_DIR = project_root / "data" / "cleaned"
INPUT_PATH = DATA_DIR / "orders_enriched_cleaned.csv"
OUTPUT_PATH = project_root / "data" / "processed" / "orders_features.csv"

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Expected cleaned file was not found: {INPUT_PATH}")

raw_df = pd.read_csv(INPUT_PATH)

# Parse time columns
for col in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date", "shipping_limit_date"]:
    if col in raw_df.columns:
        raw_df[col] = pd.to_datetime(raw_df[col], errors="coerce")

# Basic order-level preparation
raw_df = raw_df.copy()
raw_df["order_value"] = raw_df["price"].fillna(0) + raw_df["freight_value"].fillna(0)
raw_df["is_delayed"] = raw_df["delivery_delay_days"].fillna(0) > 0
raw_df["is_cancelled"] = raw_df["order_status"].astype(str).str.lower().str.contains("cancel", na=False)
raw_df["review_score"] = pd.to_numeric(raw_df["review_score"], errors="coerce")
raw_df["payment_installments"] = pd.to_numeric(raw_df["payment_installments"], errors="coerce")
raw_df["product_weight_g"] = pd.to_numeric(raw_df["product_weight_g"], errors="coerce")
raw_df["price"] = pd.to_numeric(raw_df["price"], errors="coerce")
raw_df["freight_value"] = pd.to_numeric(raw_df["freight_value"], errors="coerce")

# Fill missing values with sensible defaults
raw_df["delivery_delay_days"] = pd.to_numeric(raw_df["delivery_delay_days"], errors="coerce")
raw_df["delivery_delay_days"] = raw_df["delivery_delay_days"].fillna(raw_df["delivery_delay_days"].median())
raw_df["review_score"] = raw_df["review_score"].fillna(raw_df["review_score"].median())
raw_df["order_value"] = raw_df["order_value"].fillna(raw_df["order_value"].median())
raw_df["product_weight_g"] = raw_df["product_weight_g"].fillna(raw_df["product_weight_g"].median())
raw_df["payment_installments"] = raw_df["payment_installments"].fillna(raw_df["payment_installments"].median())
raw_df["payment_type"] = raw_df["payment_type"].fillna("unknown")
raw_df["product_category_name_english"] = raw_df["product_category_name_english"].fillna("unknown")
raw_df["city"] = raw_df["city"].fillna("unknown")
raw_df["state"] = raw_df["state"].fillna("unknown")
raw_df["seller_city"] = raw_df["seller_city"].fillna("unknown")
raw_df["seller_state"] = raw_df["seller_state"].fillna("unknown")

# Create time features
raw_df["order_purchase_date"] = raw_df["order_purchase_timestamp"].dt.date
raw_df["order_month"] = raw_df["order_purchase_timestamp"].dt.month
raw_df["order_year"] = raw_df["order_purchase_timestamp"].dt.year
raw_df["order_day_of_week"] = raw_df["order_purchase_timestamp"].dt.dayofweek
raw_df["order_hour"] = raw_df["order_purchase_timestamp"].dt.hour
raw_df["is_weekend"] = raw_df["order_day_of_week"].isin([5, 6]).astype(int)

# Aggregate to one row per order
order_level = (
    raw_df.groupby("order_id", as_index=False)
    .agg(
        customer_id=("customer_id", "first"),
        order_status=("order_status", "first"),
        order_purchase_timestamp=("order_purchase_timestamp", "first"),
        order_month=("order_month", "first"),
        order_year=("order_year", "first"),
        order_day_of_week=("order_day_of_week", "first"),
        order_hour=("order_hour", "first"),
        is_weekend=("is_weekend", "first"),
        payment_type=("payment_type", "first"),
        city=("city", "first"),
        state=("state", "first"),
        product_category_name_english=("product_category_name_english", "first"),
        seller_id=("seller_id", "first"),
        seller_city=("seller_city", "first"),
        seller_state=("seller_state", "first"),
        price=("price", "mean"),
        freight_value=("freight_value", "mean"),
        order_value=("order_value", "sum"),
        item_count=("order_item_id", "count"),
        delivery_delay_days=("delivery_delay_days", "mean"),
        product_weight_g=("product_weight_g", "mean"),
        review_score=("review_score", "mean"),
        payment_installments=("payment_installments", "mean"),
        is_delayed=("is_delayed", "max"),
        is_cancelled=("is_cancelled", "max"),
    )
)

# Customer-level aggregates
customer_features = (
    order_level.groupby("customer_id", as_index=False)
    .agg(
        customer_order_count=("order_id", "count"),
        customer_avg_order_value=("order_value", "mean"),
        customer_avg_delivery_delay=("delivery_delay_days", "mean"),
        customer_late_rate=("is_delayed", "mean"),
        customer_cancel_rate=("is_cancelled", "mean"),
        customer_avg_review_score=("review_score", "mean"),
    )
)

# Seller-level aggregates
seller_features = (
    order_level.groupby("seller_id", as_index=False)
    .agg(
        seller_order_count=("order_id", "count"),
        seller_avg_order_value=("order_value", "mean"),
        seller_avg_delivery_delay=("delivery_delay_days", "mean"),
        seller_late_rate=("is_delayed", "mean"),
        seller_cancel_rate=("is_cancelled", "mean"),
        seller_avg_review_score=("review_score", "mean"),
    )
)

# Merge features into order rows
features_df = order_level.merge(customer_features, on="customer_id", how="left")
features_df = features_df.merge(seller_features, on="seller_id", how="left")

# Add a simple target for downstream modeling
features_df["target_delayed"] = features_df["is_delayed"].astype(int)

# Convert categorical columns to numeric indicators
categorical_cols = [
    "payment_type",
    "city",
    "state",
    "product_category_name_english",
    "seller_city",
    "seller_state",
    "order_status",
]
for col in categorical_cols:
    if col in features_df.columns:
        features_df[col] = features_df[col].astype(str)

feature_frame = pd.get_dummies(features_df, columns=categorical_cols, drop_first=False)

# Keep the most useful columns and drop any remaining identifier columns
model_columns = [
    c for c in feature_frame.columns
    if c not in ["order_id", "customer_id", "seller_id", "order_purchase_timestamp", "order_purchase_date"]
]
processed_df = feature_frame[model_columns].copy()

# Ensure the target remains available and output directory exists
processed_df["target_delayed"] = processed_df["target_delayed"].astype(int)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
processed_df.to_csv(OUTPUT_PATH, index=False)

print(f"Feature engineering complete. Shape: {processed_df.shape}")
print(f"Saved to: {OUTPUT_PATH}")
print(processed_df.head(2).to_string())


Feature engineering complete. Shape: (99441, 4897)
Saved to: C:\Users\sreey\OneDrive\Desktop\WI integration -YTOR\data\processed\orders_features.csv
   order_month  order_year  order_day_of_week  order_hour  is_weekend  price  freight_value  order_value  item_count  delivery_delay_days  product_weight_g  review_score  payment_installments  is_delayed  is_cancelled  customer_order_count  customer_avg_order_value  customer_avg_delivery_delay  customer_late_rate  customer_cancel_rate  customer_avg_review_score  seller_order_count  seller_avg_order_value  seller_avg_delivery_delay  seller_late_rate  seller_cancel_rate  seller_avg_review_score  target_delayed  payment_type_boleto  payment_type_credit_card  payment_type_debit_card  payment_type_not_defined  payment_type_unknown  payment_type_voucher  city_abadia dos dourados  city_abadiania  city_abaete  city_abaetetuba  city_abaiara  city_abaira  city_abare  city_abatia  city_abdon batista  city_abelardo luz  city_abrantes  city_abre campo 